> **교육생용 정답 코드**
> Python 3 커널을 선택하고 위에서부터 셀을 실행하세요.

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams["axes.unicode_minus"] = False
plt.rc("font", family="Malgun Gothic")


# [Project] 공항기상관측(AMOS) 기상청 데이터 분석

---

## 프로젝트 목표

- 공항기상관측(AMOS) 기상청 데이터 분석 프로젝트를 통해 시간에 따른 공항의 풍속, 강수량, 기온 데이터를 분석하고 통찰을 얻는 능력을 배양합니다.

---

## 프로젝트 목차
1. **데이터 읽기:** 공항기상관측(AMOS) 기상청 데이터를 불러오고 DataFrame 구조를 확인<br>
    1.1. 데이터 불러오기<br>
    1.2. 데이터 확인하기<br>
<br>
2. **데이터 정제:** 데이터 확인 후 형 변환 및 이상치 데이터 처리<br>
    2.1. 김포공항 데이터만 추출<br>
    2.2. 결측치 제거<br>
    2.3. 단위 변환<br>
<br>
3. **데이터 시각화:** 시각화를 통해 데이터 특성 파악<br>
    3.1. 풍속 변화량 확인<br>
    3.2. 기온 변화량 시각화<br>
    3.3. 강수량 시각화<br>

---

## 데이터 출처
- 출처: 기상청 기상자료개방포털 (URL: https://data.kma.go.kr/data/air/selectAmosRltmList.do?pgmNo=575)
- 수집 기간: 2024년 1월 1일 ~ 2024년 11월 30일
- 수집 대상: 7개 공항의 풍향, 풍속, 기온, 이슬점온도, 해면기압, 현지기압, 강수량

---

## 1. 데이터 읽기

필요한 패키지 설치 및 `import`한 후 `pandas`를 사용하여 데이터를 읽고 어떠한 데이터가 저장되어 있는지 확인합니다.

### 1.1. 데이터 불러오기

In [ ]:
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt

먼저, 공항기상관측(AMOS) 기상청 데이터를 불러와 `amos_all`에 저장합니다.

In [ ]:
# pd.read_csv를 통하여 공항기상관측(AMOS) 기상청 데이터를 데이터프레임 형태로 읽어옵니다.
amos_all = pd.read_csv("amos_rawdata.csv")

In [ ]:
# 공항기상관측(AMOS) 상위 5개 데이터를 출력합니다.
amos_all.head()

측정 시점을 나타내는 `일시` 컬럼은 `datetime` 형식으로 변환하고, 데이터의 구조를 확인합니다.

In [ ]:
amos_all["일시"] = pd.to_datetime(amos_all["일시"])

# 데이터프레임 정보를 요약하여 출력합니다. 
amos_all.info()

### 1.2. 데이터 확인하기

불러온 데이터프레임의 특정 컬럼 데이터를 **중복없이 오름차순 정렬하여** 확인해 봅시다.

In [ ]:
unique_time_list = sorted(list(set(amos_all['일시']))) # 측정 일시 종류 확인
print(unique_time_list) # AMOS 측정 일시 데이터 확인
print(len(set(amos_all['일시']))) # 측정 일시 종류 확인

1시간 단위로 측정한 데이터이며, 총 8038 회차 측정한 데이터임을 알 수 있습니다.

다음으로, 측정한 지점을 확인해 봅시다.

In [ ]:
# amos_all DataFrame 지점명 데이터 확인
print(sorted(list(set(amos_all['지점명'])))) # AMOS 측정 지점명 데이터 확인
print(len(set(amos_all['지점명']))) # 측정 지점명 종류 확인

총 7개 공항에서 측정한 데이터임을 확인할 수 있습니다.

다음으로, 각 측정 데이터의 통계 수치를 확인해 봅시다.

In [ ]:
# amos_all DataFrame 통계수치 확인
amos_all.describe()

#### 퀴즈 1. 측정 데이터 중 강수량의 평균값과 표준 편차를 각각 `precipitation_mean`, `precipitation_std` 변수에 저장하세요.

In [ ]:
precipitation = amos_all["강수량(mm)"]
precipitation_mean = precipitation.mean()
precipitation_std = precipitation.std()

--- 

## 2. 데이터 정제

In [ ]:
print("측정 시작일: ", amos_all['일시'].iloc[0]) # 정렬되어 있으므로 0번째 데이터가 시작일
print("측정 종료일: ", amos_all['일시'].iloc[-1]) # 정렬되어 있으므로 마지막 데이터가 종료일

데이터를 확인해 보니 2024년 1월부터 11월까지 수집된 데이터인 것을 알 수 있습니다.

이번 프로젝트에서는 **김포공항**의 1년 풍속, 기온, 강수량의 변화 추이를 시각화하여 비교하고자 합니다.

### 2.1. 김포공항 데이터만 추출

수집된 데이터 중 김포공항 데이터만 추출하여 `amos_gimpo`에 저장합니다.

In [ ]:
amos_gimpo = amos_all[amos_all['지점명'] == '김포공항'] # 김포공항 데이터만 추출

In [ ]:
amos_gimpo

`지점` column과 `지점명` column은 같은 정보를 다르게 표현한 값입니다. 둘 중 하나는 삭제합니다.

In [ ]:
# 불필요한 지점 컬럼 제거
amos_gimpo = amos_gimpo.drop(['지점'], axis=1)
# 측정 일시에 따라 정렬
amos_gimpo.sort_values(by='일시', inplace=True)

amos_gimpo

### 2.2. 결측치 제거

정제한 데이터를 확인해보면, `강수량(mm)` 컬럼에 `NaN` 값이 많이 있는 것을 알 수 있습니다. 이는 결측이 누락된 결측치를 의미하며, 데이터의 특성에 맞춰 처리해줘야 합니다.

강수량의 값이 결측된 경우는 비가 오지 않아 강수량을 측정할 수 없는 경우일 가능성이 높으므로, 이번 프로젝트에서는 강수량의 결측치를 0으로 대체합니다.

In [ ]:
# 강수량의 결측치를 0으로 대체
amos_gimpo.loc[amos_gimpo['강수량(mm)'].isnull(), '강수량(mm)'] = 0

In [ ]:
# 결측치 개수 확인
amos_gimpo.isnull().sum()

### 2.3. 단위 변환

데이터 중 `풍속(KT)` 컬럼은 단위가 익숙한 `m/s` 대신 `KT` (노트) 로 되어 있는 것을 알 수 있습니다.

이대로 사용해도 되지만, 데이터를 이해하기 쉽도록 `m/s`로 변환해줍니다.

여기서는 1 노트 = 1.852 km/h = 0.5144 m/s 로 변환합니다.

In [ ]:
def knot_to_ms(knot):
    return knot * 0.514444

amos_gimpo['풍속(m/s)'] = knot_to_ms(amos_gimpo['풍속(KT)'])

#### 퀴즈 2. 양양공항에서 측정한 풍속 값의 평균을 계산하여 `yangyang_wind_speed_mean` 변수에 저장하세요.
- 단위는 `m/s`로 변환합니다.

In [ ]:
yangyang_wind = amos_all.loc[amos_all["지점명"].eq("양양공항"), "풍속(KT)"]
yangyang_wind_speed_mean = knot_to_ms(yangyang_wind).mean()

---

## 3. 데이터 시각화

김포공항 데이터만 추출한 `amos_gimpo` 데이터를 사용하여 시간에 따른 풍속, 기온, 강수량의 분포와 변화 추이를 시각화합니다.

### 3.1 풍속 변화량 확인

`풍속(m/s)` 컬럼을 시각화하여 풍속 변화량을 확인합니다.

In [ ]:
# 측정 시간에 따른 풍속 변화 확인
plt.figure(figsize=(15, 5))


plt.plot(amos_gimpo['일시'], amos_gimpo['풍속(m/s)'])

시각화 결과, 풍속이 주로 2m/s 정도인 것으로 추정되나, 데이터가 너무 많아 오히려 풍속의 변화량을 파악하기 어렵습니다.

이동 평균을 활용하여 좀 더 알아보기 쉽도록 풍속의 변화량을 시각화합니다.

In [ ]:
# 측정 시간에 따른 기온 변화 확인 (전체 지점)
# 일반 plot과 이동 평균을 같이 확인

plt.figure(figsize=(15, 5))

window_size = 120 # 5일 치 데이터를 이용하여 이동 평균 계산

plt.plot(amos_gimpo['일시'], amos_gimpo['풍속(m/s)'], alpha=0.5)
plt.plot(amos_gimpo['일시'], amos_gimpo['풍속(m/s)'].rolling(window=window_size).mean())

이동 평균 데이터를 확인해 보니 풍속이 2m/s ~ 4m/s 정도에서 주로 변화하고 있음을 알 수 있고, 측정 시간에 따른 변화는 크지 않음을 알 수 있습니다.

### 3.2. 기온 변화량 시각화

`기온(°C)` 컬럼을 시각화하여 시간에 따른 기온 변화를 확인합니다.

In [ ]:
# 측정 시간에 따른 기온 변화 확인
plt.figure(figsize=(15, 5))

plt.plot(amos_gimpo['일시'], amos_gimpo['기온(°C)'])

그래프를 확인했을 때, 겨울철 (11월 ~ 1월) 기온이 여름철 기온보다 훨씬 낮고, 계절의 변화에 따라 기온도 변화하는 추세를 보이는 것을 알 수 있습니다.

기온뿐만 아니라 이슬점 온도도 비슷한 추세를 보이는지 확인해 봅시다.

In [ ]:
plt.figure(figsize=(15, 5))

plt.plot(amos_gimpo['일시'], amos_gimpo['기온(°C)'])
plt.plot(amos_gimpo['일시'], amos_gimpo['이슬점온도(°C)'])

이슬점 온도는 기온 보다는 조금 낮지만, 계절에 따라 비슷한 추세를 보이는 것을 알 수 있습니다.

### 3.3. 강수량 시각화

기온을 시각화 한 것과 비슷하게, 강수량도 시간에 따라 변화하는 추세를 보이는지 확인해 봅시다.

In [ ]:
plt.figure(figsize=(15, 5))

plt.plot(amos_gimpo['일시'], amos_gimpo['강수량(mm)'])

여름철에 비가 많이 오는 것 같긴 하지만, 강수량이 0인 경우가 훨씬 많아서 직관적으로 보이지는 않습니다. 

시간에 따른 변화를 시각화 하기 보다는, 특정 기간 내 강수량의 분포를 시각화 하는 것이 더 직관적일 것 같습니다.

In [ ]:
# 여름철 데이터

amos_gimpo['일시(월)'] = amos_gimpo['일시'].dt.month

amos_gimpo_summer = amos_gimpo[(amos_gimpo['일시(월)'] >= 6) & (amos_gimpo['일시(월)'] <= 8)]
amos_gimpo_winter = amos_gimpo[(amos_gimpo['일시(월)'] >= 12) | (amos_gimpo['일시(월)'] <= 2)]

여름철 데이터와 겨울철 데이터를 분리했으니, 히스토그램을 그려서 분포를 비교해봅시다.

In [ ]:
plt.figure(figsize=(15, 5))

plt.hist(amos_gimpo_summer['강수량(mm)'], bins=100, alpha=0.5, label='Summer', color = 'r')  
plt.hist(amos_gimpo_winter['강수량(mm)'], bins=100, alpha=0.5, label='Winter', color = 'b')

plt.legend()

강수량이 0인 경우가 너무 많아서 여전히 직관적이지 않습니다.

강수량이 0보다 큰 데이터만 필터링 해서 시각화 해보겠습니다.

In [ ]:
amos_gimpo_summer.loc[amos_gimpo_summer['강수량(mm)'] > 0]

In [ ]:
plt.figure(figsize=(15, 5))


plt.hist(amos_gimpo_summer.loc[amos_gimpo_summer['강수량(mm)'] > 0]['강수량(mm)'], alpha=0.5, label='Summer', color = 'r',range=(0,30), bins=50)
plt.hist(amos_gimpo_winter.loc[amos_gimpo_winter['강수량(mm)'] > 0]['강수량(mm)'], alpha=0.5, label='Winter', color = 'b',range=(0,30), bins=50)

plt.legend()

겨울철 보다 여름철에 강수량이 더 넓게 분포하는 것을 확인할 수 있습니다.

#### 퀴즈 3-1. **김포공항** 에서 측정된 강수량이 0보다 큰 데이터의 갯수를 `rainy_day_count` 변수에 저장하세요.
- 퀴즈 2와 달리 양양공항 데이터를 사용하지 않음에 유의합니다.

In [ ]:
rainy_day_count = int(amos_gimpo["강수량(mm)"].gt(0).sum())

#### 퀴즈 3-2. 김포공항의 여름철 (6월 ~ 8월) 강수량의 평균을 계산하여 `summer_precipitation_mean` 변수에 저장하세요.
- 이때, 강수량이 0인 경우는 제외합니다.

In [ ]:
summer_precipitation = amos_gimpo_summer["강수량(mm)"]
summer_precipitation_mean = summer_precipitation[summer_precipitation.gt(0)].mean()

## 정답 확인

In [ ]:
ans = {
    "퀴즈1": (float(precipitation_mean), float(precipitation_std)),
    "퀴즈2": float(yangyang_wind_speed_mean),
    "퀴즈3": (float(rainy_day_count), float(summer_precipitation_mean)),
}

print(ans)